# Cuba water geographic names

Explore the names of Cuba’s streams, canals, ditches, lakes, lagoons, reservoirs,
bays and other water features. Each consolidated name object has one ID, preferred
name, string-array alternatives, detailed feature type, preferred-name source
reference and one Point location.

Run the cells in order from `examples/`, using **GeoPandas**, **Pandas** and **PyArrow**.
The previews show five records rather than listing the complete registry.


## Load the data

Read [Water Geographic Names](../data/water-geographic-names/) as GeoParquet.
Each geometry is one **Point in EPSG:4326**, WGS 84 longitude and latitude.
A name’s Point locates the object; it does not describe the extent of a river or
water-body polygon. Names remain in the list even when no physical feature has
an assigned name ID.


In [1]:
from pathlib import Path
import json
import geopandas as gpd

ROOT = Path("..")
names = gpd.read_parquet(ROOT / "data/water-geographic-names/cuba_water_geographic_names.parquet")
print(f"Rows: {len(names):,}")
print(f"Columns: {len(names.columns)}")


Rows: 9,343
Columns: 6


## Fields

- `id`: consolidated name object identifier.
- `preferred_name`: selected, type-complete label.
- `alternative_names`: a JSON array of strings belonging to this object.
- `object_type`: detailed feature type, such as stream, canal, lagoon or reservoir.
- `source_observations`: source references for the preferred name only; `other` means no documented source, with no invented source ID.
- `geometry`: one Point from the preferred-name source. Recorded coordinates are used where available; otherwise a representative point comes from the original source geometry. Selection details remain in the workflow audit.

See [fields.json](../data/water-geographic-names/fields.json) for definitions.


In [2]:
structure = names.dtypes.astype(str).rename_axis("Column").reset_index(name="Data type")
print(structure.to_string(index=False))


             Column Data type
                 id       str
     preferred_name       str
  alternative_names       str
        object_type       str
source_observations       str
           geometry  geometry


## Preview the names

Alternative names contain strings, without source records or object references.
Decode them with `json.loads(...)`.


In [3]:
preview = names[["preferred_name", "object_type", "alternative_names"]].head().copy()
preview["alternative_names"] = preview["alternative_names"].map(lambda values: "; ".join(json.loads(values)))
preview = preview.rename(columns={"preferred_name": "Geographic name", "object_type": "Feature type", "alternative_names": "Alternative names"})
print(preview.to_string(index=False))


          Geographic name Feature type Alternative names
         Embalse Jucarral    reservoir                  
         Embalse La Ceiba    reservoir                  
Laguna de Mala Habitación       lagoon                  
           Embalse Caonao    reservoir      Presa Caonao
        Embalse Voladoras    reservoir                  


## Count feature types

The detailed types distinguish rivers and canals from lakes, reservoirs, bays
and other features.


In [4]:
counts = names["object_type"].fillna("Unresolved type").value_counts().rename_axis("Feature type").reset_index(name="Name objects")
print(counts.to_string(index=False))


              Feature type  Name objects
                    stream          4922
                       bay           474
                      cove           421
                 reservoir           362
                     canal           357
                    lagoon           355
            marine channel           292
                     water           281
                      cape           211
                      pond           186
                     drain           172
                     ditch           154
                     shoal           148
                      lake           113
               tidal creek           104
             swimming pool            88
                      reef            71
                   estuary            70
                   channel            67
                     inlet            63
              stream mouth            59
                     marsh            56
                 waterfall            37
                

## Use the names with physical features

[Water-body Geometry](../data/water-body-geometry/) and
[River Centerlines](../data/river-centerlines/) link directly to this list:
join their `gazetteer_id` to the name object’s `id` to obtain the preferred name,
alternatives and type. Name objects contain no reverse geometry references.

The physical datasets retain their own geometry and naming status. A null name
ID does not remove an unnamed physical feature.

Continue with [the hydrography map](hydrography_map.ipynb) or the
[ONEI river systems example](onei_river_systems.ipynb).
